# 04 — Training

Train SpecTNT with/without CTL loss using 4-fold CV.

## Setup

In [ ]:
import sys, os, json, copy, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
sys.path.insert(0, os.path.abspath(".."))

from utils.label_conversion import CLASSES
from utils.dataset import HarmonixDataset, CHUNK_SECONDS, HOP_SECONDS, TARGET_FPS
from utils.augmentations import default_augment
from utils.spectnt import SpecTNT
from utils.losses import combined_loss, CombinedWithCTLLoss

device = torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: xpu


## Load all song IDs

In [ ]:
import pandas as pd
META_PATH = os.path.abspath("../data/harmonixset/dataset/metadata.csv")
meta = pd.read_csv(META_PATH, encoding="utf-8", encoding_errors="replace")
all_song_ids = meta["File"].tolist()
print(f"Total songs: {len(all_song_ids)}")


Total songs: 912


## 4-fold CV split

In [ ]:
from sklearn.model_selection import KFold
kfold = KFold(n_splits=4, shuffle=True, random_state=42)
folds = list(kfold.split(all_song_ids))
for i, (train_idx, val_idx) in enumerate(folds):
    print(f"Fold {i+1}: {len(train_idx)} train, {len(val_idx)} val")


Fold 1: 684 train, 228 val
Fold 2: 684 train, 228 val
Fold 3: 684 train, 228 val
Fold 4: 684 train, 228 val


## Training configuration

In [ ]:
BATCH_SIZE = 32  # 128 OOMs on 8GB XPU; increase if you have more VRAM
BATCHES_PER_EPOCH = 500
N_EPOCHS = 100
PATIENCE = 2
LR = 0.0005
WEIGHT_DECAY = 0.9
CLIP_GRAD_NORM = 5.0

SAVE_DIR = "../checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {N_EPOCHS}")
print(f"LR: {LR}, WD: {WEIGHT_DECAY}")


Batch size: 32
Max epochs: 100
LR: 0.0005, WD: 0.9


## Custom collate for variable-length token sequences

In [ ]:
def custom_collate(batch):
    chunks = torch.stack([item[0] for item in batch], 0)
    b_target = torch.stack([item[1] for item in batch], 0)
    f_target = torch.stack([item[2] for item in batch], 0)
    tokens = [item[3] for item in batch]
    return chunks, b_target, f_target, tokens


## Training function

In [ ]:
def train_fold(fold_idx, train_ids, val_ids, use_ctl=False):
    train_ds = HarmonixDataset(train_ids, augment=default_augment())
    val_ds = HarmonixDataset(val_ids)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=custom_collate)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=custom_collate)

    model = SpecTNT(dim=96, n_blocks=5).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    if use_ctl:
        criterion = CombinedWithCTLLoss()
    else:
        criterion = combined_loss

    best_loss = float("inf")
    patience_counter = 0
    accum_steps = 4  # gradient accumulation to simulate batch=128

    for epoch in range(N_EPOCHS):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        progress = tqdm(range(BATCHES_PER_EPOCH), desc=f"Fold {fold_idx+1} E{epoch+1}")
        for step in progress:
            try:
                batch = next(iter(train_loader))
            except StopIteration:
                break
            chunks, b_target, f_target, tokens = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
            b_logits, f_logits = model(chunks)

            if use_ctl:
                B = b_logits.shape[0]
                T = b_logits.shape[1]
                il = torch.full((B,), T, dtype=torch.long, device=device)
                tl = torch.tensor([len(t) for t in tokens], dtype=torch.long, device=device)
                loss, ctl_loss = criterion(b_logits, b_target, f_logits, f_target, tokens, il, tl)
            else:
                loss = criterion(b_logits, b_target, f_logits, f_target)

            loss = loss / accum_steps
            loss.backward()
            train_loss += loss.item() * accum_steps

            if (step + 1) % accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
                optimizer.step()
                optimizer.zero_grad()
                if hasattr(torch, "xpu") and torch.xpu.is_available():
                    torch.xpu.empty_cache()

            progress.set_postfix(loss=loss.item() * accum_steps)
        train_loss /= BATCHES_PER_EPOCH

        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for chunks, b_target, f_target, tokens in val_loader:
                chunks = chunks.to(device)
                b_target = b_target.to(device)
                f_target = f_target.to(device)
                b_logits, f_logits = model(chunks)
                if use_ctl:
                    B = b_logits.shape[0]
                    T = b_logits.shape[1]
                    il = torch.full((B,), T, dtype=torch.long, device=device)
                    tl = torch.tensor([len(t) for t in tokens], dtype=torch.long, device=device)
                    loss, ctl_loss = criterion(b_logits, b_target, f_logits, f_target, tokens, il, tl)
                else:
                    loss = criterion(b_logits, b_target, f_logits, f_target)
                if isinstance(loss, tuple):
                    loss = loss[0]
                val_loss += loss.item() * chunks.shape[0]
                n_val += chunks.shape[0]
        val_loss /= max(n_val, 1)

        scheduler.step()
        if hasattr(torch, "xpu") and torch.xpu.is_available():
            torch.xpu.empty_cache()

        variant = "ctl" if use_ctl else "base"
        print(f"  Val loss: {val_loss:.4f}  (best: {best_loss:.4f})")

        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            ckpt_path = os.path.join(SAVE_DIR, f"spectnt_{variant}_fold{fold_idx+1}.pt")
            torch.save({
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "fold": fold_idx,
                "epoch": epoch,
                "val_loss": val_loss,
            }, ckpt_path)
            print(f"  Checkpoint saved: {ckpt_path}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    print(f"Fold {fold_idx+1} done. Best val loss: {best_loss:.4f}")
    return best_loss


## Train Variant A (No CTL)

In [ ]:
# WARNING: This will take several hours.
# Uncomment and run to start training.
# 
# for fold_idx, (train_idx, val_idx) in enumerate(folds):
#     train_ids = [all_song_ids[i] for i in train_idx]
#     val_ids = [all_song_ids[i] for i in val_idx]
#     train_fold(fold_idx, train_ids, val_ids, use_ctl=False)


## Train Variant B (With CTL)

In [ ]:
# WARNING: This will take several hours.
# Uncomment and run to start training.
# 
for fold_idx, (train_idx, val_idx) in enumerate(folds):
    train_ids = [all_song_ids[i] for i in train_idx]
    val_ids = [all_song_ids[i] for i in val_idx]
    train_fold(fold_idx, train_ids, val_ids, use_ctl=True)


Fold 1 E1: 100%|██████████| 500/500 [13:08<00:00,  1.58s/it, loss=0.787]


  Val loss: 0.7960  (best: inf)
  Checkpoint saved: ../checkpoints\spectnt_ctl_fold1.pt


Fold 1 E2: 100%|██████████| 500/500 [12:23<00:00,  1.49s/it, loss=0.686]


  Val loss: 0.7201  (best: 0.7960)
  Checkpoint saved: ../checkpoints\spectnt_ctl_fold1.pt


Fold 1 E3: 100%|██████████| 500/500 [12:14<00:00,  1.47s/it, loss=0.684]


...

  Val loss: 0.7003  (best: 0.7090)
  Checkpoint saved: ../checkpoints\spectnt_ctl_fold4.pt


Fold 4 E5: 100%|██████████| 500/500 [12:15<00:00,  1.47s/it, loss=0.677]


  Val loss: 0.7053  (best: 0.7003)


Fold 4 E6: 100%|██████████| 500/500 [12:12<00:00,  1.47s/it, loss=0.606]


  Val loss: 0.7123  (best: 0.7003)
  Early stopping at epoch 6
Fold 4 done. Best val loss: 0.7003


## Quick sanity: overfit a single batch

In [ ]:
print("Sanity: overfit a single batch")
ds = HarmonixDataset(all_song_ids[:5])
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=custom_collate)
batch = next(iter(loader))
chunks, b_target, f_target, tokens = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
model = SpecTNT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for i in range(50):
    b_logits, f_logits = model(chunks)
    loss = combined_loss(b_logits, b_target, f_logits, f_target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (i+1) % 10 == 0:
        print(f"  Step {i+1}: loss = {loss.item():.4f}")


Sanity: overfit a single batch
  Step 10: loss = 3.6807
  Step 20: loss = 0.1413
  Step 30: loss = 0.0552
  Step 40: loss = 0.0341
  Step 50: loss = 0.0209
